# TRIE — IDD perception fine-tune

Fine-tunes YOLOv11 on the India Driving Dataset so `ai/perception/engine.py` moves from a COCO-pretrained baseline (no auto-rickshaw class, Western car-dominated distribution) to a real, measured Indian-road accuracy number. Source spec: `ai/training/kaggle_idd_notebook.md` in the TRIE repo — this notebook is that spec, made pushable via the Kaggle API rather than pasted by hand.

In [ ]:
!pip -q install ultralytics
import yaml, os
ROOT = "/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset"
names = ["animal","autorickshaw","bicycle","bus","car","caravan","motorcycle",
         "person","rider","traffic light","traffic sign","trailer","train",
         "truck","vehicle fallback"]
yaml.safe_dump({"path": ROOT, "train": "train/images", "val": "val/images",
                "test": "test/images", "nc": len(names), "names": names},
               open("/kaggle/working/idd.yaml", "w"))
print(open("/kaggle/working/idd.yaml").read())

In [ ]:
from ultralytics import YOLO
# yolo11s, COCO-pretrained: Ultralytics remaps the 8 shared classes (car/bus/
# truck/bicycle/motorcycle/person/train + one) into the 15-class head by name,
# a warm start. Full 33.6k images, 640px, 40 epochs — a T4 handles batch 16.
YOLO("yolo11s.pt").train(
    data="/kaggle/working/idd.yaml",
    epochs=40, imgsz=640, batch=16, patience=12,
    project="/kaggle/working/runs", name="perception_idd",
    fliplr=0.5, flipud=0.0, exist_ok=True,
)

In [ ]:
import json
m = YOLO("/kaggle/working/runs/perception_idd/weights/best.pt").val(data="/kaggle/working/idd.yaml")
result = {
    "mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
    "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
    "per_class": {n: {"mAP50": round(float(m.box.ap50[i]), 4)}
                  for i, n in enumerate(names) if i < len(m.box.ap50)},
}
json.dump(result, open("/kaggle/working/evaluation.json", "w"), indent=2)
print(json.dumps(result, indent=2))

In [ ]:
# Copy the two artifacts TRIE needs out of the run directory and to the
# notebook's own output root, so `kaggle kernels output` picks them up
# without having to know the nested runs/ path.
import shutil
shutil.copy("/kaggle/working/runs/perception_idd/weights/best.pt", "/kaggle/working/best.pt")
shutil.copy("/kaggle/working/evaluation.json", "/kaggle/working/evaluation.json")
print("done — best.pt and evaluation.json are in /kaggle/working/")